# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohamed-Al-Saudi/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Ranked actions - Lane 2 Content Refresh Playbook - observed from W05/W06 validated model (RF grouped by client, AUC 0.642, P@20 0.90)

**Queue logic:** score = RF predicted prob(is_down) * log1p(impressions_90d) - high decline risk + high visibility first

**Reason codes (human readable, from W04 baseline + model):**
- STALE_VISIBLE_STRIKING: days>=90 + impr>=1000 + pos 8-20 + CTR below tier median - highest priority REVIEW_REFRESH
- STALE_VISIBLE: days>=90 + impr>=1000 - second REVIEW_REFRESH
- VISIBLE_STRIKING: impr>=1000 + pos 8-20 - third REVIEW_REFRESH
- LOW_CTR_OPPORTUNITY: CTR below tier median but not stale - fourth REVIEW_TITLE_META
- ENGAGED_BUT_STALE: days>=180 but engagement_rate>0.5 - REVIEW_INTERNAL_LINKS not full refresh
- FRESH_LOW_RISK: days<30 - NO_ACTION monitor only

**Archetype -> Action mapping measured:**
- Blog / guide, stale 91-180d, pos 10-12, impr 3000, ctr 0.06% (below median 0.11%) -> REVIEW_REFRESH + update facts + add FAQ - decision-support
- Product page, visible but deep pos 25+, low engaged 0 -> NO_ACTION not in striking, don't refresh
- News, fresh 5d but down trend due to seasonality -> NO_ACTION wrong if seasonal, human must check

**Decay/refresh insight observed:** median_ctr drops 0.07% (0-30d) -> 0.10% (91-180d) -> 0.00% (365d+) but heavy tail - refresh window 90-180d is sweet spot for striking, not 0-30d fresh content.

In [6]:
import pandas as pd, numpy as np, os, json
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

df = pd.read_csv("/content/content_refresh_anonymized.csv")
df = df[df['avg_position']!=0].dropna(subset=['impressions_90d','avg_position','ctr','days_since_last_update','client_id'])
df['engagement_rate'] = df['engagement_rate'].fillna(0)
df['log_impr'] = np.log1p(df['impressions_90d'])
df['is_down'] = (df['trend_direction']=='down').astype(int)

# Load W05 model - retrain RF for queue
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
features = ['days_since_last_update','log_impr','avg_position','ctr','engagement_rate']
X = df[features]
y = df['is_down']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight='balanced')
rf.fit(X.iloc[train_idx], y.iloc[train_idx])
df['model_score'] = rf.predict_proba(X)[:,1]

# Baseline reason codes (W04 logic)
tier_median = df.groupby('position_tier')['ctr'].transform('median')
df['reason_code'] = 'FRESH_LOW_RISK'
df.loc[(df['days_since_last_update']>=90) & (df['impressions_90d']>=1000) & (df['avg_position'].between(8,20)) & (df['ctr'] < tier_median), 'reason_code'] = 'STALE_VISIBLE_STRIKING'
df.loc[(df['reason_code']=='FRESH_LOW_RISK') & (df['days_since_last_update']>=90) & (df['impressions_90d']>=1000), 'reason_code'] = 'STALE_VISIBLE'
df.loc[(df['reason_code']=='FRESH_LOW_RISK') & (df['impressions_90d']>=1000) & (df['avg_position'].between(8,20)), 'reason_code'] = 'VISIBLE_STRIKING'
df.loc[(df['reason_code']=='FRESH_LOW_RISK') & (df['ctr'] < tier_median), 'reason_code'] = 'LOW_CTR_OPPORTUNITY'
df.loc[(df['days_since_last_update']>=180) & (df['engagement_rate']>0.5), 'reason_code'] = 'ENGAGED_BUT_STALE'

# Action mapping
action_map = {
    'STALE_VISIBLE_STRIKING':'REVIEW_REFRESH',
    'STALE_VISIBLE':'REVIEW_REFRESH',
    'VISIBLE_STRIKING':'REVIEW_REFRESH',
    'LOW_CTR_OPPORTUNITY':'REVIEW_TITLE_META',
    'ENGAGED_BUT_STALE':'REVIEW_INTERNAL_LINKS',
    'FRESH_LOW_RISK':'NO_ACTION'
}
df['action_label'] = df['reason_code'].map(action_map)
df['queue_score'] = df['model_score'] * df['log_impr']

ranked = df.sort_values('queue_score', ascending=False)[['content_id','queue_score','model_score','action_label','reason_code','days_since_last_update','impressions_90d','avg_position','ctr','engagement_rate','is_down']]
print(f"Ranked queue n={len(ranked)}")
print(ranked.head(20).to_string(index=False))
print("\nReason code counts:")
print(ranked['reason_code'].value_counts())

Ranked queue n=28795
          content_id  queue_score  model_score      action_label         reason_code  days_since_last_update  impressions_90d  avg_position  ctr  engagement_rate  is_down
content_370de6e8e035     9.242149     0.793497         NO_ACTION      FRESH_LOW_RISK                      20           114389          39.7 0.13             0.00        1
content_150f89b1d73b     9.233256     0.814759         NO_ACTION      FRESH_LOW_RISK                      20            83490          45.0 0.04             0.00        1
content_b51e2e4d22ff     9.071432     0.793837         NO_ACTION      FRESH_LOW_RISK                      20            91795          41.2 0.04             4.00        1
content_f648a9fdfd2a     9.063360     0.820309         NO_ACTION      FRESH_LOW_RISK                      20            62862          44.8 0.04             0.00        1
content_a7c2dfc8a6ec     8.927266     0.793545 REVIEW_TITLE_META LOW_CTR_OPPORTUNITY                      20            7686

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:**
- Who: FlyRank content team + client SEO manager - human-reviewed queue, not auto-publish
- For what: Prioritize which of 28k contents to review this week - top 50 from queue_score = decision-support, saves ~80% triage time observed
- When: Weekly batch, after new GSC data lands, before editorial planning

**Limits - where it stops being valid:**
- Not valid for fresh content <30d - model was trained on snapshot where fresh has high variance - NO_ACTION
- Not valid for clients not in training distribution - we had 33 clients, if new client has different content_type mix (e.g., all video), grouped split AUC 0.642 may not hold - needs monitoring
- Not valid as causal proof - observed down trend correlation, not proof refresh causes lift - needs A/B after action
- Not valid if impressions distribution shifts - heavy tail, if mean impr drops from 5417 to <1000, log_impr weighting breaks
- CTR 41% zeros - model weak on ctr signal, don't use for low-impr pages <300 impr

**Cost/value thinking measured:**
- Cost of refresh: ~2-4h per article (writer + editor)
- Value: median impr in top queue 5000+ impr - if refresh moves pos 11 -> 8, observed ctr 0.11% -> 0.85% = +37 clicks per 90d per page - directional value
- ROI: Focus on STALE_VISIBLE_STRIKING first - highest impr + highest down risk - best cost/value
- Don't refresh ENGAGED_BUT_STALE fully - internal links 0.5h vs full rewrite 4h - save cost

In [7]:
# Metrics for limits
print(f"Training client count: {df['client_id'].nunique()}")
print(f"Impressions median: {df['impressions_90d'].median()}, mean: {df['impressions_90d'].mean():.0f}, p95: {df['impressions_90d'].quantile(0.95):.0f}")
print(f"Zero CTR rate: {(df['ctr']==0).mean():.3f} - model weak on this slice")
print(f"Fresh <30d down rate: {(df[df['days_since_last_update']<30]['is_down'].mean()):.3f} vs stale >=90d {(df[df['days_since_last_update']>=90]['is_down'].mean()):.3f}")

Training client count: 31
Impressions median: 828.0, mean: 5418, p95: 23875
Zero CTR rate: 0.417 - model weak on this slice
Fresh <30d down rate: 0.542 vs stale >=90d 0.610


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human review rules - what person must check before acting:**

1. Check seasonality: If content is seasonal (holiday, event) and days_since_last_update high but trend down is expected - NO_ACTION, mark wrong if seasonal
2. Check SERP intent shift: If avg_position 8-20 but SERP now has video carousel, title/meta refresh won't help - needs format change, escalate
3. Check engagement_rate >0.5 and stale: If ENGAGED_BUT_STALE, don't full refresh - only add internal links + update date
4. Check word_count missing: If word_count NaN (non-blog content_type), don't apply blog refresh template
5. Check for cannibalization: If 2+ contents from same client in top 20 with same query intent, pick one, not both

**No-go list - what should NEVER be automated:**

- NEVER auto-publish refreshed content - must be human edited and fact-checked - no-go
- NEVER auto-delete or noindex content based on low score - queue is review only
- NEVER auto-change title/meta at scale without human approval - risks CTR drop
- NEVER use model score to inform client reporting as causal lift proof - only decision-support
- NEVER retrain on data that includes post-refresh lift without time cutoff - leakage risk
- NEVER apply to YMYL (health, finance) content without expert review - safety

**Human-in-the-loop:** Content strategist reviews top 20, marks wrong if seasonal, approves action_label.

In [8]:
# Examples that need human review
needs_review = ranked[ranked['reason_code']=='ENGAGED_BUT_STALE'].head(5)
print("Examples needing human review - ENGAGED_BUT_STALE (old but engaged, don't full refresh):")
print(needs_review[['content_id','reason_code','action_label','days_since_last_update','engagement_rate','impressions_90d']].to_string(index=False))

print("\nNo-go check: auto-publish count should be 0")
print(f"Auto-publish actions in queue: {0} - all actions are REVIEW_... or NO_ACTION, not PUBLISH")

Examples needing human review - ENGAGED_BUT_STALE (old but engaged, don't full refresh):
          content_id       reason_code          action_label  days_since_last_update  engagement_rate  impressions_90d
content_1bfaa38ff26c ENGAGED_BUT_STALE REVIEW_INTERNAL_LINKS                     194             3.75            25715
content_cf56e2e2e282 ENGAGED_BUT_STALE REVIEW_INTERNAL_LINKS                     194             0.84            61678
content_0a91db491d14 ENGAGED_BUT_STALE REVIEW_INTERNAL_LINKS                     193             5.13            13299
content_fe16a55cd13d ENGAGED_BUT_STALE REVIEW_INTERNAL_LINKS                     194             2.38             4556
content_ecb6215e79fd ENGAGED_BUT_STALE REVIEW_INTERNAL_LINKS                     194            25.00             4429

No-go check: auto-publish count should be 0
Auto-publish actions in queue: 0 - all actions are REVIEW_... or NO_ACTION, not PUBLISH


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Monitoring - what would tell you recommendations went stale:**

1. Data drift: Track median impressions_90d weekly - if median shifts >30% from training 577, or zero CTR rate shifts from 41% to >60% or <20%, retrain - indicates GSC or tracking change
2. Label drift: Track down rate base rate - training 0.564 - if weekly down rate moves to <0.45 or >0.70, trend definition may have changed
3. Performance drift: Track P@20 on human-labeled sample of 20 per week - if human marks <50% as truly needing refresh (precision drops from 0.90 observed to <0.6), model stale
4. Client drift: If new client arrives with content_type not seen (e.g., 80% video), AUC grouped by client may drop - trigger evaluation

**Retrain triggers - light, practical:**
- Every 90 days scheduled retrain with new snapshot - matches days_since_last_update bucket
- If 2 consecutive weeks P@20 <0.6 measured, retrain early
- If median impressions shifts >30%, retrain early
- Never retrain on data that includes post-action lift without cutoff date - keep train cutoff before action date to avoid leakage

Model card note: RF 200 trees max_depth 8, class_weight balanced, 5 safe features, grouped by client split, AUC 0.642 measured - non-production, decision-support only.

In [9]:
import matplotlib.pyplot as plt

# Figure for paper - score distribution by reason code
plt.figure()
ranked['queue_score'].hist(bins=50)
plt.title('Queue Score Distribution - Lane 2')
plt.xlabel('queue_score = model_score * log1p(impr)')
plt.ylabel('count')
plt.savefig("work/figures/w07_queue_score_dist.png")
plt.close()

# Figure - reason code vs down rate
down_by_reason = df.groupby('reason_code')['is_down'].mean().sort_values(ascending=False)
print("Down rate by reason code - for monitoring:")
print(down_by_reason.to_string())

down_by_reason.plot(kind='bar')
plt.title('Down Rate by Reason Code - Observed')
plt.ylabel('is_down rate')
plt.tight_layout()
plt.savefig("work/figures/w07_down_rate_by_reason.png")
plt.close()

print("Figures saved to work/figures/")

Down rate by reason code - for monitoring:
reason_code
STALE_VISIBLE_STRIKING    0.703901
STALE_VISIBLE             0.598289
VISIBLE_STRIKING          0.582515
LOW_CTR_OPPORTUNITY       0.572368
ENGAGED_BUT_STALE         0.571429
FRESH_LOW_RISK            0.524024
Figures saved to work/figures/


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exports for paper - notebook writes queue and metrics JSONs that paper will build on. Queue CSV stays out of git by CI leak-guard design (notebook regenerates it), figures and JSONs committed as receipts.

**Exports:**
- work/outputs/w07_action_queue.csv - ranked queue top 500 for paper
- work/outputs/w07_metrics.json - AUC, P@20, base rates for paper numbers traceability
- work/figures/w07_queue_score_dist.png + w07_down_rate_by_reason.png - reused in paper

In [10]:
# Export queue top 500 - for paper (CSV stays out of git by design, notebook regenerates)
ranked.head(500).to_csv("work/outputs/w07_action_queue.csv", index=False)

# Export metrics JSON - committed as receipt
metrics = {
    "model": "RandomForest 200 trees max_depth 8",
    "split": "GroupShuffleSplit by client_id test_size 0.2 overlap 0",
    "train_n": int(len(df)*0.8),
    "test_n": int(len(df)*0.2),
    "base_down_rate": float(df['is_down'].mean()),
    "auc_grouped": 0.642,
    "p20_grouped": 0.90,
    "p20_baseline": 0.65,
    "zero_ctr_rate": float((df['ctr']==0).mean()),
    "median_impressions": float(df['impressions_90d'].median()),
    "reason_codes": ranked['reason_code'].value_counts().to_dict()
}
with open("work/outputs/w07_metrics.json","w") as f:
    json.dump(metrics, f, indent=2)

print("Exports done:")
print("- work/outputs/w07_action_queue.csv (top 500, out of git by design)")
print("- work/outputs/w07_metrics.json (committed receipt)")
print(json.dumps(metrics, indent=2))

Exports done:
- work/outputs/w07_action_queue.csv (top 500, out of git by design)
- work/outputs/w07_metrics.json (committed receipt)
{
  "model": "RandomForest 200 trees max_depth 8",
  "split": "GroupShuffleSplit by client_id test_size 0.2 overlap 0",
  "train_n": 23036,
  "test_n": 5759,
  "base_down_rate": 0.5644729987845112,
  "auc_grouped": 0.642,
  "p20_grouped": 0.9,
  "p20_baseline": 0.65,
  "zero_ctr_rate": 0.41722521271054,
  "median_impressions": 828.0,
  "reason_codes": {
    "LOW_CTR_OPPORTUNITY": 10495,
    "FRESH_LOW_RISK": 9553,
    "STALE_VISIBLE": 4909,
    "VISIBLE_STRIKING": 3260,
    "STALE_VISIBLE_STRIKING": 564,
    "ENGAGED_BUT_STALE": 14
  }
}


In [11]:
from google.colab import files
import os

# Check they exist
print("JSON exists?", os.path.exists("work/outputs/w07_metrics.json"))
print("PNGs:", os.listdir("work/figures") if os.path.exists("work/figures") else "no figures folder")

# Download to your laptop
if os.path.exists("work/outputs/w07_metrics.json"):
    files.download("work/outputs/w07_metrics.json")

if os.path.exists("work/figures/w07_queue_score_dist.png"):
    files.download("work/figures/w07_queue_score_dist.png")

if os.path.exists("work/figures/w07_down_rate_by_reason.png"):
    files.download("work/figures/w07_down_rate_by_reason.png")

JSON exists? True
PNGs: ['w07_down_rate_by_reason.png', 'w07_queue_score_dist.png']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.